In [1]:
# ===== EXPERIMENT CONFIGURATION =====

PROJECT_NAME = "Road Traffic Detection using Drone (ML)"
MODEL_NAME = "yolov12s"

DATASET_YAML = r"D:\ML Dataset\dataset.yaml"
DATASET_ROOT = r"D:\ML Dataset\FINAL_DATASET_PROCESSED"

OUTPUT_DIR = r"D:\Projects\Road Traffic Detection using Drone (ML)\Outputs-Results"
RUN_NAME = "yolov12"

IMG_SIZE = 640
EPOCHS = 35
BATCH = 4
WORKERS = 4
SEED = 42
PATIENCE = 10

CACHE = True
DEVICE = 0

In [2]:
from ultralytics import YOLO
import torch
import os
import pandas as pd
import time

In [3]:
assert torch.cuda.is_available(), "CUDA NOT AVAILABLE — STOP"

print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)

GPU: NVIDIA GeForce RTX 2050
Torch: 2.5.1+cu121


In [4]:
train_images = os.path.join(DATASET_ROOT, "images/train")
train_labels = os.path.join(DATASET_ROOT, "labels/train")

val_images = os.path.join(DATASET_ROOT, "images/val")
val_labels = os.path.join(DATASET_ROOT, "labels/val")

print("Train Images:", len(os.listdir(train_images)))
print("Train Labels:", len(os.listdir(train_labels)))

print("Val Images:", len(os.listdir(val_images)))
print("Val Labels:", len(os.listdir(val_labels)))

assert len(os.listdir(train_images)) == len(os.listdir(train_labels)), "Train mismatch"
assert len(os.listdir(val_images)) == len(os.listdir(val_labels)), "Val mismatch"

Train Images: 20000
Train Labels: 20000
Val Images: 5000
Val Labels: 5000


In [5]:
model = YOLO("yolo12s.pt")

In [7]:
CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR,
    RUN_NAME,
    "weights",
    "last.pt"
)

resume_training = False

if os.path.exists(CHECKPOINT_PATH):
    print("Checkpoint found → Resuming training")
    resume_training = True
else:
    print("No checkpoint → Fresh training")

No checkpoint → Fresh training


In [8]:
train_args = dict(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    patience=PATIENCE,
    cache=CACHE,

    project=OUTPUT_DIR,
    name=RUN_NAME,
    exist_ok=True,

    amp=True,          # critical for RTX 2050
    verbose=True
)

In [9]:
start_time = time.time()

if resume_training:
    model = YOLO(CHECKPOINT_PATH)

    results = model.train(
        resume=True,
        **train_args
    )

else:
    results = model.train(
        **train_args
    )

end_time = time.time()

print(f"Total Training Time: {(end_time - start_time)/3600:.2f} hours")

New https://pypi.org/project/ultralytics/8.4.23 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ML Dataset\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov12, nbs

In [10]:
results_csv = os.path.join(OUTPUT_DIR, RUN_NAME, "results.csv")

assert os.path.exists(results_csv), "results.csv not found"

df = pd.read_csv(results_csv)
df.tail()

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2,lr/pg3,lr/pg4,lr/pg5,lr/pg6,lr/pg7
30,31,39362.5,0.82647,0.43521,0.82511,0.79252,0.67625,0.73280,0.53923,0.85539,0.48330,0.86081,0.004543,0.001514,0.004543,0.001514,0.004543,0.001514,0.004543,0.001514
31,32,40547.4,0.81388,0.42698,0.82334,0.78800,0.68827,0.73668,0.54168,0.84940,0.48074,0.85897,0.003694,0.001231,0.003694,0.001231,0.003694,0.001231,0.003694,0.001231
32,33,41765.4,0.80402,0.42036,0.82137,0.77497,0.69633,0.73802,0.54363,0.84481,0.47771,0.85866,0.002846,0.000949,0.002846,0.000949,0.002846,0.000949,0.002846,0.000949
33,34,43178.0,0.79026,0.41203,0.81952,0.79720,0.68300,0.73678,0.54378,0.84200,0.47498,0.85813,0.001997,0.000666,0.001997,0.000666,0.001997,0.000666,0.001997,0.000666
34,35,44617.6,0.78059,0.40434,0.81780,0.80544,0.67730,0.73710,0.54533,0.83995,0.47359,0.85778,0.001149,0.000383,0.001149,0.000383,0.001149,0.000383,0.001149,0.000383


In [11]:
BEST_MODEL = os.path.join(
    OUTPUT_DIR,
    RUN_NAME,
    "weights",
    "best.pt"
)

assert os.path.exists(BEST_MODEL), "Best model not found"

print("Best model:", BEST_MODEL)

Best model: D:\Projects\Road Traffic Detection using Drone (ML)\Outputs-Results\yolov12\weights\best.pt


In [12]:
model = YOLO(BEST_MODEL)

metrics = model.val()

metrics

Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
YOLOv12s summary (fused): 159 layers, 9,233,202 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 277.8169.9 MB/s, size: 174.7 KB)
val: Scanning D:\ML Dataset\FINAL_DATASET_PROCESSED\labels\val.cache... 5000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
val: D:\ML Dataset\FINAL_DATASET_PROCESSED\images\val\UAVDT_YOLO_M0606_img001213.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 3.3it/s 1:34<0.3ss
                   all       5000      79843      0.809      0.679      0.739      0.547
                   car        888       7244      0.844      0.678      0.768      0.526
                   bus       3832      63461      0.972       0.97      0.986      0.788
                 truck       1926       3996      0.846      0.795      0.862   

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001E82FDBB5D0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
 

In [13]:
model_size = os.path.getsize(BEST_MODEL) / (1024 * 1024)

print(f"Model Size: {model_size:.2f} MB")

Model Size: 18.05 MB


In [14]:
import time
import glob

val_images_sample = glob.glob(os.path.join(val_images, "*.jpg"))[:200]

model = YOLO(BEST_MODEL)

start = time.time()

for img in val_images_sample:
    model.predict(img, imgsz=640, verbose=False)

end = time.time()

total_time = end - start
fps = len(val_images_sample) / total_time

print(f"Inference Time per image: {total_time / len(val_images_sample):.4f} sec")
print(f"FPS: {fps:.2f}")

Inference Time per image: 0.0295 sec
FPS: 33.89


In [15]:
from ultralytics.utils.torch_utils import model_info

model = YOLO(BEST_MODEL)

model_info(model.model, verbose=True)

YOLOv12s summary: 272 layers, 9,255,458 parameters, 0 gradients, 21.5 GFLOPs


(272, 9255458, 0, 21.532825600000002)